# Sıfırdan Dil Modeli · Bölüm 5
## Dikkat: kelimeler birbirine bakıyor

Geçen bölümde her token'a bir vektör verdik. Ama o vektör yalnızca kendi kelimesini bilir: gömme tablosu kelimeye bakar, cümleye bakmaz. Bu defterde **öz-dikkatin** (*self-attention*) en basit hâlini yazıyoruz: her kelime cümledeki bütün kelimelere bakar ve onlardan bir parça alır. Öğrenilen hiçbir ağırlık yok; yalnızca nokta çarpım, softmax ve ağırlıklı toplam.

Adımlar Sebastian Raschka'nın *Build a Large Language Model (From Scratch)* kitabının 3.3. bölümünü izler. Kod, kitabın açık kaynak deposundaki kodun ([rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch), Apache-2.0) Türkçe bir cümle için uyarlamasıdır.

Hücreleri sırayla çalıştırın: hücreye tıklayın, **Shift + Enter**'a basın.

## 1. Cümle ve vektörleri
Kaşağı'dan altı kelimelik bir cümle. Her kelimeye üç sayılık bir vektör veriyoruz. Gerçek modelde bu vektörler gömme katmanından gelir ve çok daha uzundur (Çırak'ta 256 sayı); buradaki üç sayıyı biz seçtik, hesabı gözle izleyebilelim diye. Kabaca birinci sayı "kişi", ikincisi "eşya", üçüncüsü "yer" gibi düşünülebilir; gerçek modelde sayıların böyle adları yoktur.

In [ ]:
import torch

cumle = "Babam yalağın içinde kırılmış kaşağıyı gördü"
kelimeler = cumle.split()
print(kelimeler)

In [ ]:
inputs = torch.tensor(
  [[0.90, 0.10, 0.20],   # Babam
   [0.10, 0.40, 0.80],   # yalağın
   [0.10, 0.20, 0.70],   # içinde
   [0.20, 0.90, 0.30],   # kırılmış
   [0.10, 0.80, 0.30],   # kaşağıyı
   [0.70, 0.30, 0.20]]   # gördü
)
print(inputs.shape)

## 2. Skor: nokta çarpım
Bir kelime seçelim: **kaşağıyı**. Python saymaya sıfırdan başladığı için beşinci kelime `inputs[4]`. Ona *sorgu* (query) diyoruz. Sorgunun her kelimeyle **nokta çarpımı**, o kelimeye ne kadar bakacağının ham ölçüsü: dikkat skoru. `enumerate` bir listeyi sırayla gezerken her elemanın sıra numarasını da verir.

In [ ]:
query = inputs[4]                    # kaşağıyı
attn_scores_5 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_5[i] = torch.dot(x_i, query)
print(attn_scores_5)

Nokta çarpım iki vektörü eleman eleman çarpıp toplar. Elle yapalım ve `torch.dot` ile karşılaştıralım (kırılmış · kaşağıyı):

In [ ]:
res = 0.
for idx, element in enumerate(inputs[3]):
    res += inputs[3][idx] * query[idx]
print(res)
print(torch.dot(inputs[3], query))

## 3. Ağırlıklar: softmax
Skorları, toplamı 1 olan **dikkat ağırlıklarına** çeviriyoruz. En basit yol, her skoru toplama bölmek:

In [ ]:
attn_weights_5_tmp = attn_scores_5 / attn_scores_5.sum()
print("Ağırlıklar:", attn_weights_5_tmp)
print("Toplam:", attn_weights_5_tmp.sum())

Pratikte **softmax** kullanılır: her skorun `exp`'i (e üzeri skor) alınır, sonra toplama bölünür. e üzeri her sayı pozitiftir; ağırlıklar da hep pozitif olur.

In [ ]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_5_naive = softmax_naive(attn_scores_5)
print("Ağırlıklar:", attn_weights_5_naive)
print("Toplam:", attn_weights_5_naive.sum())

Neden softmax? Gerçek vektörlerde skor eksi de çıkabilir. Bir skoru eksi olan üç sayıyla iki yolu karşılaştıralım:

In [ ]:
ornek = torch.tensor([2.0, -1.0, 0.5])
print("Toplama böl:", ornek / ornek.sum())
print("Softmax:    ", softmax_naive(ornek))

PyTorch'un kendi softmax'ı aynı sonucu verir; çok büyük ya da çok küçük sayılarda da taşmadan çalışır. `dim=0`, işlemin hangi boyut boyunca yapılacağını söyler.

In [ ]:
attn_weights_5 = torch.softmax(attn_scores_5, dim=0)
print("Ağırlıklar:", attn_weights_5)
print("Toplam:", attn_weights_5.sum())

## 4. Bağlam vektörü
Her kelimenin vektörünü ağırlığıyla çarpıp topluyoruz. Çıkan **bağlam vektörü**, kaşağıyı'nın cümledeki öteki kelimelerden aldığı payları taşır.

In [ ]:
context_vec_5 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_5 += attn_weights_5[i] * x_i
print(context_vec_5)

## 5. Hepsi birden: matris
Aynı hesabı her kelime için yapalım: her kelimenin her kelimeyle skoru, 6 × 6'lık bir tablo.

In [ ]:
attn_scores = torch.empty(6, 6)
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)
print(attn_scores)

İki döngü yerine tek satır: `@` matris çarpımı, `.T` matrisin devriği (satırlar sütun olur). `inputs @ inputs.T` bütün nokta çarpımları bir kerede hesaplar.

In [ ]:
attn_scores = inputs @ inputs.T
print(attn_scores)

## 6. Ağırlık matrisi ve ısı haritası
Softmax'ı her satıra ayrı ayrı uyguluyoruz: `dim=-1`, son boyut, yani satır boyunca. Her satırın toplamı 1 olmalı.

In [ ]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)
print("Satır toplamları:", attn_weights.sum(dim=-1))

Sayılar yerine renkler: bir **ısı haritası**. Her satır bir kelimenin nereye baktığını gösterir. `matplotlib`, Python'un çizim kütüphanesi.

In [ ]:
import matplotlib.pyplot as plt

def isi_haritasi(agirliklar, etiketler):
    plt.figure(figsize=(6, 5))
    plt.imshow(agirliklar, cmap="viridis")
    plt.xticks(range(len(etiketler)), etiketler, rotation=45)
    plt.yticks(range(len(etiketler)), etiketler)
    plt.colorbar()
    plt.show()

isi_haritasi(attn_weights, kelimeler)

Bütün bağlam vektörleri de tek bir matris çarpımı. Beşinci satır, az önce döngüyle bulduğumuz vektör olmalı:

In [ ]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)
print("Önceki bağlam vektörü:", context_vec_5)

## 7. Çırak'ın gerçek gömmeleri
Oyuncak vektörleri biz seçtik. Aynı hesabı gerçek bir modelin gömmeleriyle yapalım: serinin modeli **Çırak**'ın, Türkçe Vikipedi'de eğitilmiş token gömmeleri (4.096 × 256). Önceki bölümlerin kodu `onceki_bolumler.py` dosyasında; Bölüm 2'de yazdığımız `BPETokenizer` oradan geliyor. Üç dosyayı serinin deposundan indiriyoruz:

In [ ]:
import os
import urllib.request

DEPO = "https://raw.githubusercontent.com/kocahmet1/sifirdan-dil-modeli/main/"

def indir(path):
    dosya = os.path.basename(path)
    if not os.path.exists(dosya):
        urllib.request.urlretrieve(DEPO + path, dosya)
    print(dosya, os.path.getsize(dosya) // 1024, "KB")

for path in ["onceki_bolumler.py", "veri/cirak_tokenizer.json",
             "veri/cirak_gomme.pt"]:
    indir(path)

In [ ]:
import json
from onceki_bolumler import BPETokenizer

with open("cirak_tokenizer.json", encoding="utf-8") as f:
    data = json.load(f)
cirak = BPETokenizer({tuple(p): 257 + i for i, p in enumerate(data["merges"])})

ids = cirak.encode(cumle)
parcalar = [cirak.decode([i]) for i in ids]
print(len(ids), parcalar)

Geçen bölümdeki `nn.Embedding` katmanı, bu sefer rastgele değil: Çırak'ın eğitilmiş tablosuyla.

In [ ]:
W = torch.load("cirak_gomme.pt").float()
gomme = torch.nn.Embedding.from_pretrained(W)
X = gomme(torch.tensor(ids))
print(W.shape, X.shape)

In [ ]:
weights = torch.softmax(X @ X.T, dim=-1)
print("Kendine:", weights.diagonal())
isi_haritasi(weights, parcalar)

## Özet
Basit öz-dikkat üç adım: her kelimenin her kelimeyle nokta çarpımı (skor), satır satır softmax (ağırlık), ağırlıklı toplam (bağlam vektörü). Matris hâlinde iki satır: `softmax(inputs @ inputs.T)` ve `attn_weights @ inputs`.

Ama bu dikkatte öğrenilen hiçbir şey yok: vektörler ne kadar benziyorsa o kadar bakıyor. Çırak'ın gerçek gömmeleriyle her parça en çok kendine bakıyor, gerisi neredeyse eşit dağılıyor. Model neye bakacağını öğrenebilmeli: sıradaki bölümde ona üç öğrenilebilir matris veriyoruz.

**Kendin dene**
- `inputs`'u 10 ile çarpıp ısı haritasını yeniden çizin: ağırlıklar ne oluyor?
- Kendi cümlenizi Çırak'ın gömmeleriyle deneyin: hangi parçalar birbirine bakıyor?

**Sıradaki bölüm:** Dikkat II · sorgu, anahtar, değer.

---
Kaynak: Sebastian Raschka, *Build a Large Language Model (From Scratch)*, Manning 2024, bölüm 3.1–3.3. Kod: [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (Apache-2.0), Türkçe uyarlama: Sıfırdan Dil Modeli. Cümle: Ömer Seyfettin, "Kaşağı" (kamu malı). Çırak'ın gömmeleri ve tokenizer'ı Türkçe Vikipedi metniyle (CC BY-SA 4.0) eğitildi.